# Analyse d'un run — expérience A (Transformer)

Charge un run (`runs/<nom>_<date>/`), recharge les meilleurs poids (`best.pt`) et analyse le modèle sur des paires de **test**. Toutes les figures sont interactives (survol, zoom, double-clic pour réinitialiser, clic sur la légende pour masquer une série).

1. entraînement · 2. performance · 3. espace latent · 4. attention · 5. importance des résidus · 6. qui interagit avec qui

Convention de couleur partout : **bleu = interaction**, **orange = pas d'interaction** ; une erreur du modèle est marquée par la forme ✕.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ppi_lm.analysis import plots
from ppi_lm.analysis.attention import pair_attention
from ppi_lm.analysis.importance import integrated_gradients
from ppi_lm.analysis.inference import (
    find_latest_run,
    load_run,
    predict_pairs,
    protein_embeddings,
    sample_pairs,
    swapped,
)
from ppi_lm.analysis.latent import amino_acid_embeddings, project_2d
from ppi_lm.analysis.network import interaction_graph, rank_partners, subgraph_around
from ppi_lm.training.metrics import binary_metrics

In [ ]:
# --- paramètres -----------------------------------------------------------
DATA_DIR = Path("../data/gold_standard_data")
# le run le plus récent qui a un best.pt (y compris un run interrompu ou `smoke`) ;
# pour un run précis : RUN_DIR = Path("../runs/transformer_A_direct_20261010-150000")
RUN_DIR = find_latest_run("../runs")
N_PAIRS = 2000  # paires de test analysées (None = tout le test, 52 048 paires)
DEVICE = "auto"  # auto -> mps sur le Mac
LATENT_METHOD = "tsne"  # pca | tsne | umap (umap : pip install umap-learn)
THEME = "light"  # light | dark
SEED = 0

plots.set_theme(THEME)

In [ ]:
run = load_run(RUN_DIR, data_dir=DATA_DIR, device=DEVICE)
n_parameters = sum(p.numel() for p in run.model.parameters())
print(f"run    : {run.run_dir.name}")
print(f"device : {run.device} | paramètres : {n_parameters:,} | best.pt = epoch {run.best_epoch}")
if run.finished:
    r = run.results
    print(f"terminé : {r['epochs_run']} epoch(s), {r['train_time_s'] / 60:.1f} min")
    if r.get("limit_eval_batches"):
        print("attention : run avec --limit-eval-batches, test évalué sur un sous-ensemble")
    keys = [
        "auroc",
        "auprc",
        "accuracy",
        "balanced_accuracy",
        "f1",
        "mcc",
        "precision",
        "recall",
        "specificity",
        "brier",
        "loss",
    ]
    table = pd.DataFrame({"validation (meilleure epoch)": r["best_val"], "test": r["test"]})
    display(table.loc[keys].style.format("{:.3f}"))
else:
    print("run interrompu (pas de results.json) : pas encore de métriques de test,")
    print("best.pt contient la meilleure epoch atteinte, les analyses ci-dessous marchent.")

# validation à chaque epoch terminée (la vue tableau des courbes)
history = pd.DataFrame(run.validation_history()).set_index("epoch")
history[["auroc", "auprc", "accuracy", "f1", "mcc", "loss", "step"]].style.format(
    {"step": "{:.0f}"}, precision=3
)

## 1. Entraînement

In [ ]:
plots.plot_training_curves(run.metrics)

In [ ]:
plots.plot_system(run.metrics)

## 2. Performance

Prédictions sur un échantillon fixe de paires de test (`N_PAIRS`). Le tableau du haut porte sur **tout** le test ; les figures sur l'échantillon.

In [ ]:
pairs = sample_pairs(run.data, "test", N_PAIRS, seed=SEED)
preds = predict_pairs(run.model, run.data, pairs)
m = binary_metrics(preds.labels, preds.logits)
print(
    f"{len(preds)} paires | AUROC {m['auroc']:.3f} | "
    f"accuracy {m['accuracy']:.3f} | MCC {m['mcc']:.3f}"
)

In [ ]:
plots.plot_roc_pr(preds, "Performance sur le test")

**Distributions** : plus les deux histogrammes sont séparés, mieux le modèle distingue. **Calibration** : quand le modèle dit 0,8, la paire interagit-elle 80 % du temps ?

In [ ]:
plots.plot_probability_distributions(preds)

In [ ]:
plots.plot_calibration(preds)

In [ ]:
plots.plot_confusion(preds)

**Longueur** : les protéines plus longues que `max_protein_length` sont coupées ; le modèle est-il moins bon sur elles ?

In [ ]:
plots.plot_metric_by_length(preds, run.data.max_protein_length)

**Symétrie** : l'entrée `A <SEP> B` n'est pas symétrique, donc p(A, B) ≠ p(B, A) en général. Moyenner les deux ordres améliore-t-il l'AUROC ?

In [ ]:
preds_swapped = predict_pairs(run.model, run.data, swapped(pairs))
plots.plot_order_symmetry(preds, preds_swapped)

## 3. Espace latent

### Acides aminés
Embedding appris pour chaque acide aminé (PCA). Si le modèle a appris de la biochimie, les acides aminés d'un même groupe se rapprochent. La classe fine est dans l'infobulle.

In [ ]:
letters, aa_vectors = amino_acid_embeddings(run.model, run.tokenizer)
plots.plot_amino_acids(letters, project_2d(aa_vectors, "pca"), "Embeddings des acides aminés")

### Protéines
Un vecteur par protéine = moyenne de ses états cachés sur toutes les paires où elle apparaît (le modèle ne voit jamais une protéine seule). Survole un point pour son identifiant UniProt.

In [ ]:
protein_ids, protein_vectors, n_pairs = protein_embeddings(preds)
protein_coords = project_2d(protein_vectors, LATENT_METHOD, seed=SEED)
lengths = np.array([len(run.data.seqs[p]) for p in protein_ids])
hover = {"protéine": protein_ids, "longueur": lengths.tolist(), "paires": n_pairs.tolist()}
plots.plot_latent(
    protein_coords,
    np.log10(lengths),
    f"Protéines ({LATENT_METHOD})",
    hover=hover,
    color_title="log10(longueur)",
)

In [ ]:
plots.plot_latent(
    protein_coords,
    n_pairs.numpy(),
    f"Protéines ({LATENT_METHOD})",
    hover=hover,
    color_title="paires dans l'échantillon",
)

### Paires
Le vecteur que voit la head de classification. Bien séparé par couleur = le modèle a trouvé une représentation utile de l'interaction ; les ✕ sont ses erreurs.

In [ ]:
pair_coords = project_2d(preds.pair_vectors, LATENT_METHOD, seed=SEED)
plots.plot_latent_pairs(pair_coords, preds, f"Paires ({LATENT_METHOD})")

## 4. Attention

La paire positive la plus sûre (vraie interaction, plus haute probabilité). À gauche toute la paire (traits = frontière A | B) ; à droite le bloc A → B : quels résidus de A regardent quels résidus de B.

In [ ]:
probs = preds.probs
positives = [i for i, (_, _, y) in enumerate(pairs) if y == 1]
best = max(positives, key=lambda i: probs[i])
pair = pairs[best]
att = pair_attention(run.model, run.data, pair)
plots.plot_attention_map(att)

In [ ]:
for layer in range(len(att.weights)):
    display(plots.plot_attention_map(att, layer=layer))

**Statistiques**, sur 20 paires positives et 20 négatives :
- *attention vers le partenaire* : part de l'attention des résidus de A qui va vers B (et inversement), par couche. Plus haute en bleu = le modèle regarde le partenaire quand il y a interaction ;
- *sinks* : attention absorbée par `<CLS>`, `<SEP>`, `<EOS>` (têtes peu informatives) ;
- *focus* : 0 = attention uniforme, 1 = sur un seul token.

In [ ]:
pos_pairs = [p for p in pairs if p[2] == 1][:20]
neg_pairs = [p for p in pairs if p[2] == 0][:20]
plots.plot_attention_statistics(
    [pair_attention(run.model, run.data, p) for p in pos_pairs],
    [pair_attention(run.model, run.data, p) for p in neg_pairs],
)

## 5. Importance des résidus (integrated gradients)

Contribution de chaque résidu au logit d'interaction (rouge : pousse vers « interaction », bleu : contre). Contrairement à l'attention, cette méthode marchera aussi pour le CNN, le BiLSTM et le BiMamba. Contrôle dans le sous-titre : somme des scores ≈ logit − référence.

In [ ]:
imp = integrated_gradients(run.model, run.data, pair, steps=64)
plots.plot_residue_importance(imp)

In [ ]:
rows = []
for segment, name in ((0, pair[0]), (1, pair[1])):
    letters, scores = imp.residue_scores(segment)
    for i in torch.argsort(scores, descending=True)[:10].tolist():
        rows.append({"protéine": name, "résidu": f"{letters[i]}{i + 1}", "IG": scores[i].item()})
pd.DataFrame(rows).style.format({"IG": "{:+.4f}"})

## 6. Qui interagit avec qui

Graphe des paires évaluées autour de la protéine la plus connectée : bleu plein = interaction trouvée, bleu pointillé = interaction ratée, orange = fausse alerte. Survole le milieu d'une arête pour sa probabilité.

In [ ]:
graph = interaction_graph(preds)
hub = max(graph.degree, key=lambda node_degree: node_degree[1])[0]
print(f"{graph.number_of_nodes()} protéines, {graph.number_of_edges()} paires | hub : {hub}")
plots.plot_interaction_graph(subgraph_around(graph, hub, radius=2), title=f"Autour de {hub}")

**Partenaires prédits** : pour cette protéine, probabilité d'interaction avec d'autres protéines du test (y compris des paires jamais listées dans le dataset). En bleu, les interactions connues.

In [ ]:
test_proteins = sorted({p for a, b, _ in run.data.pairs["test"] for p in (a, b)})
rng = np.random.default_rng(SEED)
candidates = list(rng.choice(test_proteins, size=min(300, len(test_proteins)), replace=False))
known = {b if a == hub else a for a, b, y in run.data.pairs["test"] if y == 1 and hub in (a, b)}
candidates = sorted(set(candidates) | known)
ranking = rank_partners(run.model, run.data, hub, candidates)
plots.plot_partner_ranking(ranking, known, hub)